# LLM-Hybrid Variant

Combines sentence transformer text embeddings of game content (description + mechanics) with SVD collaborative filtering scores via weighted score fusion. Content embeddings address cold-start for long-tail games with few interactions; SVD captures collaborative signal from similar users.

## 1. Load Shared Data

In [1]:
import gdown, zipfile, os

if not os.path.exists('dataset/processed'):
    FILE_ID = '1a7kCwqP2Vhlv43eep0ODdzGVR6v2ouP6'
    gdown.download(id=FILE_ID, output='processed.zip', quiet=False)
    with zipfile.ZipFile('processed.zip', 'r') as z:
        z.extractall('dataset/')
    os.remove('processed.zip')
    print('Downloaded processed data.')
else:
    print('Already exists, skipping download.')

Already exists, skipping download.


In [2]:
# Kaggle dataset setup (comment out if dataset already in dataset/ folder)
# Each team member: add kaggle_username and kaggle_api_key as Colab secrets

from google.colab import userdata
import os
os.environ['KAGGLE_USERNAME'] = userdata.get('kaggle_username')
os.environ['KAGGLE_KEY'] = userdata.get('kaggle_api_key')
!kaggle datasets download -d threnjen/board-games-database-from-boardgamegeek -p dataset/ --unzip -q
!ls dataset/

Dataset URL: https://www.kaggle.com/datasets/threnjen/board-games-database-from-boardgamegeek
License(s): CC-BY-SA-3.0
artists_reduced.csv	    mechanics.csv	      subcategories.csv
bgg_data_documentation.txt  processed		      themes.csv
designers_reduced.csv	    publishers_reduced.csv    user_ratings.csv
games.csv		    ratings_distribution.csv


In [3]:
import pandas as pd
import numpy as np
import pickle

train_df = pd.read_csv('dataset/processed/train.csv')
val_df   = pd.read_csv('dataset/processed/val.csv')
test_df  = pd.read_csv('dataset/processed/test.csv')
games_df = pd.read_csv('dataset/processed/games_cleaned.csv')
mechanics_df = pd.read_csv('dataset/mechanics.csv')

with open('dataset/processed/id_maps.pkl', 'rb') as f:
    maps = pickle.load(f)

n_users = len(maps['user2idx'])
n_items = len(maps['item2idx'])
print(f'n_users: {n_users:,} | n_items: {n_items:,}')
print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

n_users: 272,184 | n_items: 21,922
Train: 18,151,978 | Val: 272,184 | Test: 272,184


## 2. Load SVD Matrices

SVD latent factor matrices computed from the SVD baseline notebook. Reused here to provide the collaborative filtering component of the hybrid score.

In [4]:
# Download SVD matrices from shared Drive
if not os.path.exists('dataset/processed/U_svd.npy'):
    FILE_ID_SVD = '1N_U-IF-HAovnIWO62iPcIKzBsYNjguxd'
    gdown.download(id=FILE_ID_SVD, output='svd_matrices.zip', quiet=False)
    with zipfile.ZipFile('svd_matrices.zip', 'r') as z:
        z.extractall('.')
    import shutil
    shutil.copy('svd_matrices/U_svd.npy', 'dataset/processed/U_svd.npy')
    shutil.copy('svd_matrices/V_svd.npy', 'dataset/processed/V_svd.npy')
    os.remove('svd_matrices.zip')
    print('Downloaded SVD matrices.')
else:
    print('Already exists, skipping download.')

U_svd = np.load('dataset/processed/U_svd.npy')
V_svd = np.load('dataset/processed/V_svd.npy')

# Also load user mean ratings needed to add back bias when scoring
user_means = train_df.groupby('user_idx')['Rating'].mean().values

print(f'U_svd shape: {U_svd.shape} | V_svd shape: {V_svd.shape}')

Already exists, skipping download.
U_svd shape: (272184, 50) | V_svd shape: (21922, 50)


## 3. Build Item Text Embeddings

For each game, construct a text string combining the description and its mechanics list. Description captures theme and narrative; mechanics capture how the game actually plays. Both together give a more complete semantic representation of the game than either alone.

Embed all games using all-MiniLM-L6-v2  a small, fast sentence transformer trained for semantic similarity. Produces a 384-dimensional vector per game.

In [5]:
!pip install sentence-transformers -q

In [6]:
from sentence_transformers import SentenceTransformer

# Only keep games that survived filtering (have an item_idx)
active_bgg_ids = sorted(maps['item2idx'].keys())
active_games = games_df[games_df['BGGId'].isin(active_bgg_ids)].copy()
active_games['item_idx'] = active_games['BGGId'].map(maps['item2idx'])
active_games = active_games.sort_values('item_idx').reset_index(drop=True)

# Join mechanics  get mechanic column names (all columns except BGGId)
mech_cols = [c for c in mechanics_df.columns if c != 'BGGId']
active_games = active_games.merge(mechanics_df[['BGGId'] + mech_cols], on='BGGId', how='left')

# For each game, list which mechanics it has as a comma-separated string
def get_mechanics_text(row):
    mechs = [col for col in mech_cols if row.get(col, 0) == 1]
    return ', '.join(mechs) if mechs else ''

active_games['mechanics_text'] = active_games.apply(get_mechanics_text, axis=1)

# Combine description and mechanics into one text string per game
active_games['Description'] = active_games['Description'].fillna('')
active_games['game_text'] = active_games['Description'] + ' mechanics: ' + active_games['mechanics_text']

print(f'Example text for game 0:')
print(active_games['game_text'].iloc[0][:300])

Example text for game 0:
die macher game seven sequential political race different region germany player charge national political party manage limited resource help party victory win party victory point regional election different way score victory point regional election supply eighty victory point depend size region part


In [7]:
# Embed all games  this takes a few minutes on CPU, run once
model_st = SentenceTransformer('all-MiniLM-L6-v2')

texts = active_games['game_text'].tolist()
item_embeddings = model_st.encode(texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True)

# item_embeddings[i] is the embedding for item_idx i
print(f'Embeddings shape: {item_embeddings.shape}')  # (n_items, 384)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/343 [00:00<?, ?it/s]

Embeddings shape: (21922, 384)


In [8]:
# Save embeddings so we don't recompute on re-runs
np.save('dataset/processed/item_embeddings.npy', item_embeddings)
print('Saved item embeddings.')

Saved item embeddings.


## 4. Content Score

For each user, compute their taste vector as the average embedding of all games they rated in training. Then score all items by cosine similarity to this taste vector  higher similarity means the game is more semantically similar to what the user has liked.

In [9]:
from sklearn.metrics.pairwise import cosine_similarity

# Precompute: for each user, average embedding of their training items
# This is the user's content taste vector
user_taste_vectors = np.zeros((n_users, item_embeddings.shape[1]), dtype=np.float32)

for uid, group in train_df.groupby('user_idx'):
    item_idxs = group['item_idx'].values
    user_taste_vectors[uid] = item_embeddings[item_idxs].mean(axis=0)

print(f'User taste vectors shape: {user_taste_vectors.shape}')

User taste vectors shape: (272184, 384)


In [10]:
def get_content_scores(user_idx):
    # Cosine similarity between user taste vector and all item embeddings
    taste = user_taste_vectors[user_idx].reshape(1, -1)
    sims = cosine_similarity(taste, item_embeddings).flatten()
    return pd.Series(sims, index=np.arange(n_items))

## 5. SVD Score

Collaborative filtering score from the SVD baseline. Dot product of the user latent vector and all item latent vectors, with user mean rating added back.

In [11]:
def get_svd_scores(user_idx):
    scores = U_svd[user_idx] @ V_svd.T
    scores += user_means[user_idx]
    return pd.Series(scores, index=np.arange(n_items))

## 6. Hybrid Score

Weighted combination of content and SVD scores. Alpha controls how much weight goes to content vs collaborative signal. Tuned on the validation set.

In [12]:
def get_hybrid_scores(user_idx, alpha):
    content = get_content_scores(user_idx)
    svd     = get_svd_scores(user_idx)

    # Normalize both to 0-1 range before combining so scales are comparable
    content = (content - content.min()) / (content.max() - content.min() + 1e-8)
    svd     = (svd - svd.min()) / (svd.max() - svd.min() + 1e-8)

    return alpha * content + (1 - alpha) * svd

## 7. Tune Alpha on Validation Set

Try three values of alpha on the validation set to find the best content vs CF balance. We tune on validation and report final results on test  this is standard practice to avoid overfitting the hyperparameter to the test set.

In [13]:
seen = train_df.groupby('user_idx')['item_idx'].apply(set).to_dict()
# exclude val items from negative pool so they are not sampled as negatives during evaluation
val_seen = val_df.groupby('user_idx')['item_idx'].apply(set).to_dict()

def evaluate(score_series, train_df, test_df, n_negatives=99, K=10, seed=42):
    rng = np.random.default_rng(seed)
    all_items = np.array(list(maps['item2idx'].values()))
    hits, ndcgs, mrrs = [], [], []

    for _, row in test_df.iterrows():
        uid      = int(row['user_idx'])
        pos_item = int(row['item_idx'])

        user_seen  = seen.get(uid, set()) | val_seen.get(uid, set()) | {pos_item}  # fixed: val items excluded
        candidates = np.setdiff1d(all_items, list(user_seen))
        neg_items  = rng.choice(candidates, size=n_negatives, replace=False)

        items_to_rank = np.append(neg_items, pos_item)
        scores = score_series.reindex(items_to_rank).fillna(0).values
        ranked = items_to_rank[np.argsort(-scores)]

        pos_rank = np.where(ranked == pos_item)[0][0] + 1

        hits.append(1 if pos_rank <= K else 0)
        ndcgs.append(np.log(2) / np.log(pos_rank + 1) if pos_rank <= K else 0)
        mrrs.append(1 / pos_rank if pos_rank <= K else 0)

    return {
        f'HR@{K}':   round(np.mean(hits), 4),
        f'NDCG@{K}': round(np.mean(ndcgs), 4),
        f'MRR@{K}':  round(np.mean(mrrs), 4),
    }

In [14]:
# Tune on a sample of validation users for speed  tuning alpha not final eval
val_sample = val_df.sample(2000, random_state=42)

alphas = [0.3, 0.5, 0.7]
val_results = {}

for alpha in alphas:
    scores_list = []
    for uid in val_sample['user_idx'].unique():
        hybrid = get_hybrid_scores(uid, alpha)
        scores_list.append(hybrid)
    # not using evaluate() here  doing a quick HR@10 check on val sample
    print(f'alpha={alpha} tuning done')
    val_results[alpha] = evaluate(
        pd.concat(scores_list).groupby(level=0).mean(),
        train_df, val_sample
    )
    print(f'  alpha={alpha}: {val_results[alpha]}')

best_alpha = max(val_results, key=lambda a: val_results[a]['NDCG@10'])
print(f'\nBest alpha: {best_alpha}')

alpha=0.3 tuning done
  alpha=0.3: {'HR@10': np.float64(0.1685), 'NDCG@10': np.float64(0.0829), 'MRR@10': np.float64(0.0576)}
alpha=0.5 tuning done
  alpha=0.5: {'HR@10': np.float64(0.164), 'NDCG@10': np.float64(0.0792), 'MRR@10': np.float64(0.054)}
alpha=0.7 tuning done
  alpha=0.7: {'HR@10': np.float64(0.164), 'NDCG@10': np.float64(0.0788), 'MRR@10': np.float64(0.0535)}

Best alpha: 0.3


## 8. Final Evaluation on Test Set

In [15]:
print(f'Running final evaluation with alpha={best_alpha}...')

hits, ndcgs, mrrs = [], [], []
rng = np.random.default_rng(42)
all_items = np.array(list(maps['item2idx'].values()))

for _, row in test_df.iterrows():
    uid      = int(row['user_idx'])
    pos_item = int(row['item_idx'])

    # compute scores on the fly per user, no storage
    score_series = get_hybrid_scores(uid, best_alpha)

    user_seen  = seen.get(uid, set()) | val_seen.get(uid, set()) | {pos_item}
    candidates = np.setdiff1d(all_items, list(user_seen))
    neg_items  = rng.choice(candidates, size=99, replace=False)

    items_to_rank = np.append(neg_items, pos_item)
    scores = score_series.reindex(items_to_rank).fillna(0).values
    ranked = items_to_rank[np.argsort(-scores)]
    pos_rank = np.where(ranked == pos_item)[0][0] + 1

    hits.append(1 if pos_rank <= 10 else 0)
    ndcgs.append(np.log(2) / np.log(pos_rank + 1) if pos_rank <= 10 else 0)
    mrrs.append(1 / pos_rank if pos_rank <= 10 else 0)

final_results = {
    'HR@10':   round(np.mean(hits), 4),
    'NDCG@10': round(np.mean(ndcgs), 4),
    'MRR@10':  round(np.mean(mrrs), 4),
}
print(f'LLM-Hybrid (alpha={best_alpha}): {final_results}')

Running final evaluation with alpha=0.3...
LLM-Hybrid (alpha=0.3): {'HR@10': np.float64(0.2401), 'NDCG@10': np.float64(0.1259), 'MRR@10': np.float64(0.0915)}


## 9. Sample Recommendations

In [16]:
idx_to_bggid = maps['idx2item']
games_lookup = games_df.set_index('BGGId')[['Name', 'AvgRating']]

def get_recommendations(user_idx, alpha, top_k=10):
    scores = get_hybrid_scores(user_idx, alpha)
    user_seen = seen.get(user_idx, set())
    scores[list(user_seen)] = -np.inf
    top_items = scores.nlargest(top_k).index.tolist()
    bgg_ids = [idx_to_bggid[i] for i in top_items]
    return games_lookup.loc[bgg_ids].reset_index()

sample_users = test_df['user_idx'].sample(3, random_state=42).tolist()
for uid in sample_users:
    username = maps['idx2user'][uid]
    num_rated = len(train_df[train_df['user_idx'] == uid])
    print(f'\nTop 10 for user {username} | Rated {num_rated} games:')
    print(get_recommendations(uid, best_alpha).to_string(index=False))


Top 10 for user VincentParadise | Rated 7 games:
 BGGId                                             Name  AvgRating
205637                     Arkham Horror: The Card Game    8.16597
 39463                                 Cosmic Encounter    7.53816
   463                             Magic: The Gathering    7.53543
146021                                  Eldritch Horror    7.78712
162886                                    Spirit Island    8.35830
   188                                               Go    7.63525
121921 Robinson Crusoe: Adventures on the Cursed Island    7.82095
 37111             Battlestar Galactica: The Board Game    7.74169
123123                BattleCON: Devastation of Indines    7.90252
 84876                          The Castles of Burgundy    8.12711

Top 10 for user Anrab | Rated 7 games:
 BGGId                                 Name  AvgRating
 12493     Twilight Imperium: Third Edition    7.82716
 37111 Battlestar Galactica: The Board Game    7.74169
170216  